# AgroSentinela — demonstração do pipeline

**Desafio 5 — Comunicação Inteligente · InsurMinds / I2A2**

Este notebook percorre o pipeline multiagente etapa por etapa, mostrando
o que cada agente recebe e o que produz. Roda **offline**, sem chave de
API e sem internet — usando a amostra de demonstração e o gerador
climático determinístico.

Para usar dados reais do Banco Central, rode antes `python etl/run_etl.py`.


In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

import pandas as pd
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

from agrosentinela import __version__
print('AgroSentinela', __version__)


## 1. A base de dados

A carteira segue o esquema dos microdados do Sicor. A interface e este
notebook não sabem (nem precisam saber) se a origem é o dado oficial ou
a amostra de demonstração.


In [ ]:
from agrosentinela.data.loader import carregar_base, estatisticas_carteira

base = carregar_base()
print('Origem :', base.origem)
print('Aviso  :', base.aviso_procedencia)
print()
for chave, valor in estatisticas_carteira(base.carteira).items():
    print(f'{chave:>20}: {valor:,.2f}' if isinstance(valor, float) else f'{chave:>20}: {valor}')


In [ ]:
base.carteira[['municipio','uf','cultura','programa','valor_credito',
               'area_ha','tipo_seguro','dt_inic_plantio','dt_inic_colheita',
               'persona']].head(8)


## 2. Fase fenológica: o que os microdados permitem

`DT_INIC_PLANTIO` e `DT_INIC_COLHEITA` vêm da **própria operação**
registrada no Sicor. Com elas calculamos em que ponto do ciclo cada
lavoura está hoje — e é isso que diferencia «vai chover muito» de
«vai chover muito em plena colheita».


In [ ]:
from agrosentinela.rules.catalogo import get_cultura

soja = get_cultura('SOJA')
pd.DataFrame([
    {'fase': f.nome, 'início': f'{f.inicio:.0%}', 'fim': f'{f.fim:.0%}',
     **{k: v for k, v in f.sensibilidade.items()}}
    for f in soja.fases
])


## 3. O pipeline completo

Seis agentes encadeados. Forçamos um cenário de seca para que o motor de
regras tenha o que fazer — em produção, a fonte é a previsão real.


In [ ]:
from agrosentinela.orchestrator import Orquestrador
from agrosentinela.utils.logging_setup import configurar_logging

configurar_logging('WARNING')

orq = Orquestrador(base=base)
res = orq.executar(
    horizonte_dias=7,
    cenario='seca',              # None = previsão real da Open-Meteo
    intensidade_cenario=1.4,
    modo_clima='sintetico',      # 'auto' consulta a Open-Meteo
    filtros={'uf': ['MT', 'GO', 'BA']},
    usar_llm=False,              # True exige GROQ_API_KEY
    max_mensagens=8,
    respeitar_supressao=False,
    persistir=False,
)
res.rastro_df[['agente', 'duração_s', 'status', 'resumo']]


### 3.1 Etapa 2 — o que veio da fonte meteorológica


In [ ]:
res.contexto.clima_diario.head(6)


In [ ]:
res.contexto.indicadores.head(4).T.head(20)


### 3.2 Etapa 3 — eventos identificados no território


In [ ]:
res.contexto.eventos_municipio[['evento_nome','severidade','gatilhos',
                                'municipios','operacoes']].head(8)


### 3.3 Etapa 4 — alertas, com score e rastro


In [ ]:
alertas = res.alertas_df
print(f'{len(alertas)} alertas gerados')
alertas[['municipio','uf','produto','fase_nome','evento_nome',
         'severidade','score','valor_credito','tipo_seguro','regras']].head(10)


In [ ]:
# Por que este alerta existe? O rastro completo de um deles:
a = res.contexto.alertas[0]
print(f'{a.id} · {a.municipio}-{a.uf} · {a.produto} · {a.fase_nome}')
print(f'Severidade {a.severidade} · score {a.score:.0f}')
print()
for ac in a.acionamentos:
    print(f"  [{ac['regra_id']}] {ac['regra_nome']} ({ac['severidade']}, score {ac['score']:.0f})")
    print(f"      {ac['explicacao']}")
    print(f"      ação: {ac['acao'][:110]}…")
    print()


### 3.4 Etapa 5 — as mensagens geradas


In [ ]:
for m in res.contexto.mensagens[:4]:
    print('═' * 78)
    print(f'{m.persona_nome} · canal {m.canal} · {m.severidade} · score {m.score:.0f}')
    print(f'{m.caracteres}/{m.limite_canal} caracteres · origem: {m.gerado_por}')
    if m.assunto:
        print(f'Assunto: {m.assunto}')
    print('─' * 78)
    print(m.texto)
    print()


### 3.5 Etapa 6 — simulação do envio

Nenhuma mensagem é enviada de verdade. O agente reproduz o comportamento
de uma central: fila por urgência, janela de contato da persona, latência,
tentativa, falha e custo equivalente.


In [ ]:
res.envios_df[['canal','destinatário','status','motivo',
               'latência_ms','custo_estimado_R$']]


## 4. O motor de regras, isolado

As regras vivem em `rules.yaml` e podem ser avaliadas fora do pipeline —
é assim que se calibra um limiar.


In [ ]:
from agrosentinela.rules.engine import MotorRegras

motor = MotorRegras()
print(f'{len(motor.regras)} regras · versão {motor.versao}')

contexto = {
    'dias_secos_consecutivos': 22,
    'balanco_hidrico_14d': -85,
    'fase': 'floracao',
    'sensibilidade_fase': 2.0,
    'cultura': 'SOJA',
    'cultura_grupo': 'Grãos',
    'irrigada': False,
    'tem_cobertura': False,
    'valor_credito': 850_000,
    'area_ha': 180,
    'dias_para_vencimento': 60,
    'freq_evento_municipio': 0.55,
    'exposicao_municipio': 12_000_000,
}

for ac in motor.avaliar(contexto):
    print(f"{ac.regra_id:<10} {ac.severidade:<10} score {ac.score:5.1f}  {ac.regra_nome}")


## 5. O copiloto, sem alucinar número

O modelo escolhe a consulta; o pandas calcula. Sem chave de API, o
copiloto responde em modo determinístico — as ferramentas são as mesmas.


In [ ]:
from agrosentinela.agents.copiloto import CopilotoAgent

copiloto = CopilotoAgent(
    carteira=res.contexto.carteira,
    alertas=res.alertas_df,
    sinistros=base.proagro_sinistros,
    indicadores=res.contexto.indicadores,
    municipios=base.municipios,
    origem_dados=base.origem,
)

resposta = copiloto.perguntar('Quais municípios concentram os alertas críticos?')
print(resposta.texto[:900])
for t in resposta.tabelas:
    display(t.head(8))


In [ ]:
# A ferramenta também pode ser chamada diretamente:
copiloto.consultar_carteira(agrupar_por=['uf'], metricas=['n_operacoes','valor_credito','pct_cobertura'])


## 6. Comparando cenários

Cada cenário exercita um conjunto diferente de regras. É a forma de
demonstrar a solução sem depender do tempo que estiver fazendo hoje.


In [ ]:
linhas = []
for cenario in ['seca', 'geada', 'granizo', 'chuva_excessiva', 'onda_calor', 'vendaval']:
    r = orq.executar(
        horizonte_dias=7, cenario=cenario, intensidade_cenario=1.4,
        modo_clima='sintetico', filtros={'uf': ['PR', 'RS', 'MT']},
        usar_llm=False, max_mensagens=5,
        respeitar_supressao=False, persistir=False,
    )
    df = r.alertas_df
    linhas.append({
        'cenário': cenario,
        'alertas': len(df),
        'críticos': int((df['severidade'] == 'critico').sum()) if len(df) else 0,
        'exposição_R$_mi': round(df['valor_credito'].sum() / 1e6, 1) if len(df) else 0.0,
        'regras acionadas': len({x for s in df['regras'] for x in str(s).split(', ')}) if len(df) else 0,
        'mensagens': len(r.contexto.mensagens),
    })

pd.DataFrame(linhas)


## 7. Histórico do Proagro

A série de pedidos de cobertura (COP) e comprovações de perdas (RCP)
responde à pergunta que a previsão sozinha não responde: *dado que este
evento vai acontecer aqui, qual a probabilidade histórica de virar
sinistro?* É o componente `freq_evento` do score.


In [ ]:
from agrosentinela.data.loader import indices_sinistralidade

idx = indices_sinistralidade(base.proagro_sinistros)
nomes = base.municipios.set_index('cd_ibge')['municipio']
idx['municipio'] = idx['cd_ibge'].map(nomes)
idx.sort_values('n_cop', ascending=False)[
    ['municipio','evento','n_cop','n_rcp','freq_evento','taxa_acionamento']
].head(12)


---

## Próximos passos

- Interface completa: `streamlit run app.py`
- Dados oficiais do BCB: `python etl/run_etl.py --anos 2025 2026`
- Entregáveis do desafio: `python scripts/gerar_entregaveis.py`
- Testes: `pytest -q`

Licença MIT.
